# Music Recommender - Prediction/Test Notebook

Use this notebook to test song recommendations from the content-based model pipeline.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

pd.set_option('display.max_columns', None)

In [2]:
# Load cleaned dataset
df_clean = pd.read_csv('dataset_cleaned.csv')
print(f'Loaded dataset shape: {df_clean.shape}')
display(df_clean.head(3))

Loaded dataset shape: (89379, 20)


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,0,0.676,0.461,1,-6.746,0,0.1430,0.0322,0.000001,0.358,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,0,0.420,0.166,1,-17.235,1,0.0763,0.9240,0.000006,0.101,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,0,0.438,0.359,0,-9.734,1,0.0557,0.2100,0.000000,0.117,0.120,76.332,4,acoustic


In [4]:
# Build feature matrix
feature_cols = [
    'danceability', 'energy', 'tempo', 'valence', 'acousticness',
    'instrumentalness', 'liveness', 'speechiness', 'loudness',
    'duration_ms', 'popularity', 'explicit', 'key', 'mode', 'time_signature'
]
feature_cols = [c for c in feature_cols if c in df_clean.columns]

X = df_clean[feature_cols].apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.median(numeric_only=True))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Feature columns ({len(feature_cols)}): {feature_cols}')
print(f'X_scaled shape: {X_scaled.shape}')

Feature columns (15): ['danceability', 'energy', 'tempo', 'valence', 'acousticness', 'instrumentalness', 'liveness', 'speechiness', 'loudness', 'duration_ms', 'popularity', 'explicit', 'key', 'mode', 'time_signature']
X_scaled shape: (89379, 15)


In [5]:
# Train nearest-neighbor retrieval model
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(X_scaled)

search_df = df_clean.copy()
search_df['track_name_lower'] = search_df['track_name'].astype(str).str.lower().str.strip()
search_df['artists_lower'] = search_df['artists'].astype(str).str.lower().str.strip()

print('Model ready for predictions.')

Model ready for predictions.


In [6]:
def find_song_candidates(song_name, max_candidates=10):
    query = str(song_name).lower().strip()

    exact = search_df[search_df['track_name_lower'] == query]
    if not exact.empty:
        return exact[['track_name', 'artists', 'track_genre']].head(max_candidates)

    partial = search_df[search_df['track_name_lower'].str.contains(query, na=False)]
    return partial[['track_name', 'artists', 'track_genre']].drop_duplicates().head(max_candidates)


def recommend_songs(song_name, top_n=10, artist_name=None):
    query = str(song_name).lower().strip()
    candidates = search_df[search_df['track_name_lower'] == query]

    if candidates.empty:
        return {
            'status': 'not_found',
            'message': f'Song not found: {song_name}',
            'suggestions': find_song_candidates(song_name, max_candidates=10)
        }

    if artist_name is not None:
        artist_q = str(artist_name).lower().strip()
        filtered = candidates[candidates['artists_lower'].str.contains(artist_q, na=False)]
        if not filtered.empty:
            candidates = filtered

    query_index = candidates.index[0]

    distances, indices = knn_model.kneighbors(X_scaled[query_index].reshape(1, -1), n_neighbors=top_n + 1)

    rec_indices = []
    rec_distances = []
    for idx, dist in zip(indices.flatten(), distances.flatten()):
        if idx == query_index:
            continue
        rec_indices.append(idx)
        rec_distances.append(dist)
        if len(rec_indices) == top_n:
            break

    rec_df = df_clean.iloc[rec_indices][['track_name', 'artists', 'track_genre', 'popularity']].copy()
    rec_df['cosine_distance'] = rec_distances
    rec_df['similarity_score'] = 1 - rec_df['cosine_distance']
    rec_df = rec_df.sort_values('similarity_score', ascending=False).reset_index(drop=True)

    return {
        'status': 'ok',
        'query_song': df_clean.loc[query_index, ['track_name', 'artists', 'track_genre']].to_dict(),
        'recommendations': rec_df
    }

## Quick Tests

In [7]:
# Test 1
result = recommend_songs("I'm Yours", top_n=10, artist_name='Jason Mraz')

if result['status'] == 'ok':
    print('Query:', result['query_song'])
    display(result['recommendations'])
else:
    print(result['message'])
    display(result['suggestions'])

Query: {'track_name': "I'm Yours", 'artists': 'Jason Mraz', 'track_genre': 'acoustic'}


,track_name,artists,track_genre,popularity,cosine_distance,similarity_score
0,Wagon Wheel,Old Crow Medicine Show,bluegrass,67,0.026525,0.973475
1,I'm Yours,Jason Mraz,acoustic,60,0.032418,0.967582
2,The Weight - Remastered 2000,The Band,blues,72,0.051428,0.948572
3,Who You Love (feat. Katy Perry),John Mayer;Katy Perry,singer-songwriter,64,0.070597,0.929403
4,Eight Days A Week - Remastered 2009,The Beatles,british,64,0.073055,0.926945
5,スパークル,Lilas Ikuta,j-pop,68,0.073259,0.926741
6,Traingazing,Sam Wills;Honey Mooncie,chill,68,0.077893,0.922107
7,the perfect pair,beabadoobee,indie-pop,77,0.079467,0.920533
8,就是愛妳,David Tao,mandopop,58,0.081933,0.918067
9,In Case You Didn't Know,Brett Young,country,75,0.082153,0.917847


In [8]:
# Test 2
result = recommend_songs('Say Something', top_n=10)

if result['status'] == 'ok':
    print('Query:', result['query_song'])
    display(result['recommendations'])
else:
    print(result['message'])
    display(result['suggestions'])

Query: {'track_name': 'Say Something', 'artists': 'A Great Big World;Christina Aguilera', 'track_genre': 'acoustic'}


,track_name,artists,track_genre,popularity,cosine_distance,similarity_score
0,Say Something,A Great Big World;Christina Aguilera,acoustic,70,0.000797,0.999203
1,Say Something,A Great Big World;Christina Aguilera,acoustic,58,0.015474,0.984526
2,The Trouble with Wanting,Joy Williams,acoustic,60,0.015765,0.984235
3,Say Something,A Great Big World,acoustic,57,0.019130,0.980870
4,Say Something,A Great Big World,acoustic,57,0.019130,0.980870
5,I Won't Give Up,Jason Mraz,acoustic,69,0.022780,0.977220
6,Moon Song,Phoebe Bridgers,indie-pop,72,0.036997,0.963003
7,"Flightless Bird, American Mouth",Iron & Wine,folk,69,0.038599,0.961401
8,Alps,Novo Amor;Ed Tullett;Lowswimmer,ambient,61,0.042452,0.957548
9,Eu Não Sou Mais Órfão - Acústico,Gabriel Brito,brazil,54,0.042633,0.957367


In [9]:
# Custom prediction cell
song_name = 'Brave'
artist_name = 'Sara Bareilles'  # set None if not needed
top_n = 10

result = recommend_songs(song_name, top_n=top_n, artist_name=artist_name)

if result['status'] == 'ok':
    print('Query:', result['query_song'])
    display(result['recommendations'])
else:
    print(result['message'])
    print('Did you mean:')
    display(result['suggestions'])

Query: {'track_name': 'Brave', 'artists': 'Sara Bareilles', 'track_genre': 'acoustic'}


,track_name,artists,track_genre,popularity,cosine_distance,similarity_score
0,When We Stand Together,Nickelback,grunge,67,0.019245,0.980755
1,Upside Down,Paloma Faith,british,56,0.021885,0.978115
2,Siempre Brilla El Sol,Lori Meyers,spanish,56,0.031110,0.968890
3,X (feat. Maluma & Ozuna) - Remix,Nicky Jam;J Balvin;Maluma;Ozuna,latino,68,0.034085,0.965915
4,Jigsaw Falling Into Place,Radiohead,alt-rock,69,0.036185,0.963815
5,うっせぇわ,Ado,j-pop,68,0.040037,0.959963
6,Walking Away,Craig David,british,63,0.040983,0.959017
7,DLZ,TV On The Radio,r-n-b,57,0.042021,0.957979
8,Nothing In My Way,Keane,piano,64,0.043764,0.956236
9,Tamed-Dashed,ENHYPEN,anime,70,0.045062,0.954938
